# VAE Latent Market States on S&P 500 Data

This notebook extends the simple MNIST VAE intuition to **financial time-series windows**.

Instead of compressing a 28×28 image into a 2D latent vector, we compress a rolling market window into a 2D latent representation.

The goal is not direct price prediction. The goal is to ask whether the latent space learned by a VAE can represent market-state concepts such as:

- **Trend strength**
- **Volatility stress**
- **Mean-reversion pressure**
- **Macro commodity regime**

This notebook is intentionally simple and article-friendly. It can be used as a first Medium article prototype and later adapted as a component in the agri ETF / UCAIF research pipeline.

## 1. Setup

The notebook avoids `torchvision`, so it should not trigger the `torchvision::nms` issue.

Install dependencies if needed:

```bash
pip install yfinance torch scikit-learn pandas numpy matplotlib
```

In [1]:
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except Exception:
    YFINANCE_AVAILABLE = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)
print("yfinance available:", YFINANCE_AVAILABLE)

Using device: mps
yfinance available: True


## 2. Configuration

We use SPY as a tradable proxy for the S&P 500, then add macro/context assets.

Context tickers:

- `^VIX`: volatility stress proxy
- `DBC`: broad commodity proxy
- `UUP`: US dollar proxy
- `USO`: crude oil proxy
- `TLT`: long-duration bond proxy
- `GLD`: gold proxy

For your agri ETF paper, this exact structure can later be reused with `DBA`, `WEAT`, `CORN`, `SOYB`, and `CANE`.

In [2]:
@dataclass
class Config:
    market_ticker: str = "SPY"
    context_tickers: Tuple[str, ...] = ("^VIX", "DBC", "UUP", "USO", "TLT", "GLD")
    start_date: str = "2010-01-01"
    end_date: str | None = None
    window_size: int = 20
    horizon_days: int = 5
    latent_dim: int = 2
    batch_size: int = 128
    epochs: int = 40
    learning_rate: float = 1e-3
    beta_kl: float = 0.001
    train_fraction: float = 0.75
    use_synthetic_if_download_fails: bool = True

cfg = Config()
cfg

Config(market_ticker='SPY', context_tickers=('^VIX', 'DBC', 'UUP', 'USO', 'TLT', 'GLD'), start_date='2010-01-01', end_date=None, window_size=20, horizon_days=5, latent_dim=2, batch_size=128, epochs=40, learning_rate=0.001, beta_kl=0.001, train_fraction=0.75, use_synthetic_if_download_fails=True)

## 3. Data loading

The default mode downloads daily OHLCV data through `yfinance`.

If download fails, the notebook creates a synthetic market dataset so that the VAE pipeline still runs. Synthetic mode is only for debugging the mechanics.

In [3]:
def download_market_data(cfg: Config) -> pd.DataFrame:
    if not YFINANCE_AVAILABLE:
        raise RuntimeError("yfinance is not available")

    tickers = [cfg.market_ticker] + list(cfg.context_tickers)
    raw = yf.download(
        tickers,
        start=cfg.start_date,
        end=cfg.end_date,
        auto_adjust=True,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    if raw.empty:
        raise RuntimeError("No data downloaded")

    frames = []
    for ticker in tickers:
        if isinstance(raw.columns, pd.MultiIndex):
            if ticker not in raw.columns.get_level_values(0):
                continue
            temp = raw[ticker].copy()
        else:
            temp = raw.copy()

        temp.columns = [c.lower().replace(" ", "_") for c in temp.columns]
        temp["ticker"] = ticker
        temp = temp.reset_index().rename(columns={"Date": "date", "Datetime": "date"})
        frames.append(temp)

    out = pd.concat(frames, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"])
    return out


def make_synthetic_market_data(cfg: Config, n_days: int = 2600) -> pd.DataFrame:
    """Synthetic fallback: creates regime-like OHLCV for SPY and context tickers."""
    tickers = [cfg.market_ticker] + list(cfg.context_tickers)
    dates = pd.bdate_range(cfg.start_date, periods=n_days)
    rng = np.random.default_rng(RANDOM_STATE)

    regimes = rng.choice([0, 1, 2, 3], size=n_days, p=[0.45, 0.25, 0.20, 0.10])

    rows = []
    for ticker in tickers:
        price = 100.0
        for i, d in enumerate(dates):
            r = regimes[i]
            base_mu = 0.0002
            vol = 0.008

            if ticker == cfg.market_ticker:
                if r == 1:
                    mu, vol = 0.0009, 0.010
                elif r == 2:
                    mu, vol = -0.0008, 0.022
                elif r == 3:
                    mu, vol = 0.0001, 0.014
                else:
                    mu, vol = base_mu, vol
            elif ticker == "^VIX":
                mu, vol = (0.002 if r == 2 else -0.0002), 0.030
            elif ticker == "DBC":
                mu, vol = (0.0012 if r == 3 else 0.0001), 0.012
            elif ticker == "UUP":
                mu, vol = (0.0008 if r == 2 else -0.0001), 0.006
            elif ticker == "USO":
                mu, vol = (0.0015 if r == 3 else 0.0001), 0.020
            elif ticker == "TLT":
                mu, vol = (0.0008 if r == 2 else -0.0001), 0.009
            elif ticker == "GLD":
                mu, vol = (0.0007 if r in [2, 3] else 0.0001), 0.010
            else:
                mu, vol = base_mu, vol

            ret = rng.normal(mu, vol)
            open_p = price * (1 + rng.normal(0, vol / 4))
            close_p = max(1.0, open_p * (1 + ret))
            hi = max(open_p, close_p) * (1 + abs(rng.normal(0, vol / 2)))
            lo = min(open_p, close_p) * (1 - abs(rng.normal(0, vol / 2)))
            volume = int(abs(rng.normal(2_000_000, 500_000)) * (1.5 if r == 2 else 1.0))
            price = close_p
            rows.append([d, open_p, hi, lo, close_p, volume, ticker])

    return pd.DataFrame(rows, columns=["date", "open", "high", "low", "close", "volume", "ticker"])


try:
    prices = download_market_data(cfg)
    print("Downloaded real data:", prices.shape)
except Exception as e:
    print("Download failed:", repr(e))
    if cfg.use_synthetic_if_download_fails:
        prices = make_synthetic_market_data(cfg)
        print("Using synthetic fallback:", prices.shape)
    else:
        raise

prices.head()

Downloaded real data: (29120, 7)


,date,open,high,low,close,volume,ticker
0,2010-01-04,83.862049,84.623275,83.220228,84.578499,118944600.0,SPY
1,2010-01-05,84.526232,84.839678,84.220245,84.802361,111579900.0,SPY
2,2010-01-06,84.720246,85.071009,84.653082,84.862045,116074400.0,SPY
3,2010-01-07,84.705348,85.324781,84.466532,85.220299,131091100.0,SPY
4,2010-01-08,84.996405,85.541208,84.824759,85.503891,126402800.0,SPY


## 4. Feature engineering

We build features that correspond to the four interpretive concepts.

### Trend strength
Examples: 20-day return, 60-day return, moving-average slope, distance from moving average.

### Volatility stress
Examples: realized volatility, range percentage, VIX level and VIX z-score.

### Mean-reversion pressure
Examples: Bollinger z-score, RSI, distance from recent mean.

### Macro commodity regime
Examples: broad commodity momentum, dollar strength, crude oil momentum.

In [7]:
feature_df

,date,open,high,low,close,volume,ret_1d,ret_5d,ret_20d,ret_60d,...,ma_gap_20,ma_gap_60,ma_slope_20,realized_vol_20,range_pct,range_shock_20,bollinger_z_20,rsi_14,volume_z_20,forward_ret_5d
0,2010-01-04,83.862049,84.623275,83.220228,84.578499,118944600.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012353
1,2010-01-05,84.526232,84.839678,84.220245,84.802361,111579900.0,0.002647,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.007324,NaN,NaN,NaN,NaN,0.000264
2,2010-01-06,84.720246,85.071009,84.653082,84.862045,116074400.0,0.000704,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.004928,NaN,NaN,NaN,NaN,0.008003
3,2010-01-07,84.705348,85.324781,84.466532,85.220299,131091100.0,0.004222,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.010113,NaN,NaN,NaN,NaN,0.006481
4,2010-01-08,84.996405,85.541208,84.824759,85.503891,126402800.0,0.003328,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.008407,NaN,NaN,NaN,NaN,-0.008117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29115,2026-07-13,372.769989,372.890015,365.769989,367.130005,7161700.0,-0.026206,-0.039254,-0.049674,-0.165765,...,-0.029968,NaN,-0.009284,0.240909,0.018886,1.514105,-1.213262,37.131480,-0.102318,NaN
29116,2026-07-14,374.529999,376.279999,370.920013,372.149994,6303400.0,0.013674,-0.014146,-0.037228,-0.165452,...,-0.014831,NaN,-0.008734,0.247477,0.014600,1.170499,-0.605812,46.058836,-0.411667,NaN
29117,2026-07-15,372.109985,374.489990,369.500000,372.350006,7683700.0,0.000537,-0.005608,-0.061026,-0.157751,...,-0.011134,NaN,-0.008942,0.225235,0.013409,1.071014,-0.512492,55.911011,0.186907,NaN
29118,2026-07-16,367.450012,368.429993,364.049988,364.959991,9048000.0,-0.019847,-0.034957,-0.082162,-0.150406,...,-0.026537,NaN,-0.011603,0.231721,0.011763,0.939586,-1.439471,46.136675,0.674099,NaN


In [4]:
def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def add_single_ticker_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("date").copy()
    c = df["close"]
    h = df["high"]
    l = df["low"]
    v = df["volume"]

    df["ret_1d"] = c.pct_change()
    df["ret_5d"] = c.pct_change(5)
    df["ret_20d"] = c.pct_change(20)
    df["ret_60d"] = c.pct_change(60)

    df["ma_20"] = c.rolling(20).mean()
    df["ma_60"] = c.rolling(60).mean()
    df["ma_gap_20"] = c / df["ma_20"] - 1
    df["ma_gap_60"] = c / df["ma_60"] - 1
    df["ma_slope_20"] = df["ma_20"].pct_change(5)

    df["realized_vol_20"] = df["ret_1d"].rolling(20).std() * np.sqrt(252)
    df["range_pct"] = (h - l) / c.shift(1)
    df["range_shock_20"] = df["range_pct"] / df["range_pct"].rolling(20).median()

    df["bollinger_z_20"] = (c - df["ma_20"]) / c.rolling(20).std()
    df["rsi_14"] = rsi(c, 14)
    df["volume_z_20"] = (v - v.rolling(20).mean()) / v.rolling(20).std()

    df["forward_ret_5d"] = c.shift(-5) / c - 1
    return df


feature_df = prices.groupby("ticker", group_keys=False).apply(add_single_ticker_features)
feature_df.head()

,date,open,high,low,close,volume,ret_1d,ret_5d,ret_20d,ret_60d,...,ma_gap_20,ma_gap_60,ma_slope_20,realized_vol_20,range_pct,range_shock_20,bollinger_z_20,rsi_14,volume_z_20,forward_ret_5d
0,2010-01-04,83.862049,84.623275,83.220228,84.578499,118944600.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012353
1,2010-01-05,84.526232,84.839678,84.220245,84.802361,111579900.0,0.002647,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.007324,NaN,NaN,NaN,NaN,0.000264
2,2010-01-06,84.720246,85.071009,84.653082,84.862045,116074400.0,0.000704,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.004928,NaN,NaN,NaN,NaN,0.008003
3,2010-01-07,84.705348,85.324781,84.466532,85.220299,131091100.0,0.004222,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.010113,NaN,NaN,NaN,NaN,0.006481
4,2010-01-08,84.996405,85.541208,84.824759,85.503891,126402800.0,0.003328,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.008407,NaN,NaN,NaN,NaN,-0.008117


In [8]:
prices

,date,open,high,low,close,volume,ticker
0,2010-01-04,83.862049,84.623275,83.220228,84.578499,118944600.0,SPY
1,2010-01-05,84.526232,84.839678,84.220245,84.802361,111579900.0,SPY
2,2010-01-06,84.720246,85.071009,84.653082,84.862045,116074400.0,SPY
3,2010-01-07,84.705348,85.324781,84.466532,85.220299,131091100.0,SPY
4,2010-01-08,84.996405,85.541208,84.824759,85.503891,126402800.0,SPY
...,...,...,...,...,...,...,...
29115,2026-07-13,372.769989,372.890015,365.769989,367.130005,7161700.0,GLD
29116,2026-07-14,374.529999,376.279999,370.920013,372.149994,6303400.0,GLD
29117,2026-07-15,372.109985,374.489990,369.500000,372.350006,7683700.0,GLD
29118,2026-07-16,367.450012,368.429993,364.049988,364.959991,9048000.0,GLD


In [5]:
def make_wide_context(feature_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    wide_close = feature_df.pivot(index="date", columns="ticker", values="close").sort_index()
    market = feature_df[feature_df["ticker"] == cfg.market_ticker].copy().set_index("date").sort_index()

    out = pd.DataFrame(index=wide_close.index)
    out["close"] = wide_close[cfg.market_ticker]
    out["forward_ret_5d"] = market["forward_ret_5d"]

    out["trend_ret_20"] = market["ret_20d"]
    out["trend_ret_60"] = market["ret_60d"]
    out["trend_ma_gap_20"] = market["ma_gap_20"]
    out["trend_ma_slope_20"] = market["ma_slope_20"]

    out["vol_realized_20"] = market["realized_vol_20"]
    out["vol_range_shock_20"] = market["range_shock_20"]
    out["vol_range_pct"] = market["range_pct"]

    out["mr_bollinger_z_20"] = market["bollinger_z_20"]
    out["mr_rsi_14"] = market["rsi_14"]
    out["mr_ma_gap_20"] = market["ma_gap_20"]

    for ticker in cfg.context_tickers:
        if ticker in wide_close.columns:
            clean_name = ticker.replace("^", "")
            out[f"ctx_{clean_name}_ret_5d"] = wide_close[ticker].pct_change(5)
            out[f"ctx_{clean_name}_ret_20d"] = wide_close[ticker].pct_change(20)

    if "^VIX" in wide_close.columns:
        vix = wide_close["^VIX"]
        out["vix_level"] = vix
        out["vix_z_60"] = (vix - vix.rolling(60).mean()) / vix.rolling(60).std()
    else:
        out["vix_level"] = np.nan
        out["vix_z_60"] = np.nan

    out["commodity_regime_20"] = wide_close["DBC"].pct_change(20) if "DBC" in wide_close.columns else np.nan
    out["dollar_regime_20"] = wide_close["UUP"].pct_change(20) if "UUP" in wide_close.columns else np.nan
    out["oil_regime_20"] = wide_close["USO"].pct_change(20) if "USO" in wide_close.columns else np.nan

    out["trend_strength"] = out[["trend_ret_20", "trend_ma_gap_20", "trend_ma_slope_20"]].mean(axis=1)
    out["volatility_stress"] = out[["vol_realized_20", "vol_range_shock_20", "vix_z_60"]].mean(axis=1)
    out["mean_reversion_pressure"] = out[["mr_bollinger_z_20", "mr_ma_gap_20"]].mean(axis=1)
    out["macro_commodity_regime"] = out[["commodity_regime_20", "oil_regime_20"]].mean(axis=1) - out["dollar_regime_20"].fillna(0)

    return out.reset_index().dropna().sort_values("date")


market_state = make_wide_context(feature_df, cfg)
print(market_state.shape)
market_state.tail()

KeyError: 'ticker'

## 5. Build rolling windows for VAE

For images, VAE input was a flattened image.

For finance, VAE input is a flattened rolling market window:

```text
window_size × number_of_features
```

Example:

```text
20 trading days × 20 features = 400 input values
```

In [ ]:
INTERPRETATION_COLS = [
    "trend_strength",
    "volatility_stress",
    "mean_reversion_pressure",
    "macro_commodity_regime",
]

EXCLUDE_COLS = ["date", "close", "forward_ret_5d"]
FEATURE_COLS = [c for c in market_state.columns if c not in EXCLUDE_COLS]

print("Number of features:", len(FEATURE_COLS))
print(FEATURE_COLS)

In [ ]:
def make_vae_windows(df: pd.DataFrame, feature_cols: List[str], window_size: int):
    df = df.sort_values("date").reset_index(drop=True).copy()
    X = df[feature_cols].values.astype("float32")
    dates = []
    windows = []
    rows = []

    for end_idx in range(window_size - 1, len(df)):
        start_idx = end_idx - window_size + 1
        window = X[start_idx:end_idx + 1]
        windows.append(window.reshape(-1))
        dates.append(df.loc[end_idx, "date"])
        rows.append(df.loc[end_idx].copy())

    window_df = pd.DataFrame(rows).reset_index(drop=True)
    window_df["window_end_date"] = dates
    return np.array(windows, dtype="float32"), window_df


X_raw, window_meta = make_vae_windows(market_state, FEATURE_COLS, cfg.window_size)
print("Raw window matrix:", X_raw.shape)
window_meta[["date", "trend_strength", "volatility_stress", "mean_reversion_pressure", "macro_commodity_regime"]].tail()

In [ ]:
n = len(X_raw)
train_n = int(n * cfg.train_fraction)

X_train_raw = X_raw[:train_n]
X_test_raw = X_raw[train_n:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype("float32")
X_test = scaler.transform(X_test_raw).astype("float32")
X_all = scaler.transform(X_raw).astype("float32")

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train)),
    batch_size=cfg.batch_size,
    shuffle=True,
)

test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test)),
    batch_size=cfg.batch_size,
    shuffle=False,
)

input_dim = X_train.shape[1]
print("Input dimension:", input_dim)
print("Train windows:", len(X_train), "Test windows:", len(X_test))

## 6. PyTorch VAE for finance windows

Important difference from MNIST:

- MNIST pixels are in `[0, 1]`, so binary cross-entropy works.
- Finance features are standardized continuous variables, so we use **mean squared reconstruction loss**.

In [ ]:
class FinanceVAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, latent_dim: int = 2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decode(z)
        return reconstructed, mu, logvar


def vae_loss(reconstructed, x, mu, logvar, beta_kl: float = 0.001):
    recon = F.mse_loss(reconstructed, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta_kl * kl, recon, kl


model = FinanceVAE(input_dim=input_dim, hidden_dim=128, latent_dim=cfg.latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
model

In [ ]:
def train_vae(model, train_loader, epochs: int, beta_kl: float):
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = total_recon = total_kl = 0.0
        n_obs = 0

        for (x,) in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            reconstructed, mu, logvar = model(x)
            loss, recon, kl = vae_loss(reconstructed, x, mu, logvar, beta_kl=beta_kl)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_recon += recon.item()
            total_kl += kl.item()
            n_obs += x.shape[0]

        history.append({
            "epoch": epoch,
            "loss": total_loss / n_obs,
            "reconstruction": total_recon / n_obs,
            "kl": total_kl / n_obs,
        })
        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss {total_loss / n_obs:.4f} | "
                f"Recon {total_recon / n_obs:.4f} | "
                f"KL {total_kl / n_obs:.4f}"
            )
    return pd.DataFrame(history)


history = train_vae(model, train_loader, cfg.epochs, cfg.beta_kl)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history["epoch"], history["loss"], label="total")
plt.plot(history["epoch"], history["reconstruction"], label="reconstruction")
plt.plot(history["epoch"], history["kl"] * cfg.beta_kl, label="beta * KL")
plt.title("VAE training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss per window")
plt.legend()
plt.show()

## 7. Encode market windows into latent space

Each rolling market window becomes a point in 2D latent space.

If the VAE learns useful structure, this space may organize windows by trend, volatility stress, mean-reversion pressure, or macro commodity conditions.

In [ ]:
def encode_windows(model, X: np.ndarray) -> pd.DataFrame:
    model.eval()
    x = torch.tensor(X, dtype=torch.float32).to(device)
    all_mu = []
    all_logvar = []
    all_recon = []

    with torch.no_grad():
        for i in range(0, len(x), 1024):
            batch = x[i:i+1024]
            mu, logvar = model.encode(batch)
            recon = model.decode(mu)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            all_recon.append(recon.cpu())

    mu = torch.cat(all_mu).numpy()
    logvar = torch.cat(all_logvar).numpy()
    recon = torch.cat(all_recon).numpy()
    recon_error = np.mean((X - recon) ** 2, axis=1)

    out = pd.DataFrame(mu, columns=[f"z{i+1}" for i in range(mu.shape[1])])
    for i in range(logvar.shape[1]):
        out[f"z{i+1}_uncertainty"] = np.exp(logvar[:, i])
    out["reconstruction_error"] = recon_error
    return out


latent = encode_windows(model, X_all)
analysis_df = pd.concat([window_meta.reset_index(drop=True), latent], axis=1)
analysis_df[["date", "z1", "z2", "reconstruction_error"] + INTERPRETATION_COLS].tail()

## 8. Interpreting the latent dimensions

A VAE does not name latent dimensions for us. We interpret them by asking:

- Is `z1` correlated with trend strength?
- Is `z2` correlated with volatility stress?
- Do high reconstruction-error windows correspond to unusual market periods?
- Do clusters in latent space have different forward-return profiles?

In [ ]:
CORR_COLS = ["z1", "z2", "reconstruction_error"] + INTERPRETATION_COLS + ["forward_ret_5d"]
correlation_table = analysis_df[CORR_COLS].corr().loc[["z1", "z2", "reconstruction_error"], INTERPRETATION_COLS + ["forward_ret_5d"]]
correlation_table.round(3)

In [ ]:
def scatter_latent(color_col: str, title: str | None = None):
    plt.figure(figsize=(8, 6))
    s = plt.scatter(
        analysis_df["z1"],
        analysis_df["z2"],
        c=analysis_df[color_col],
        cmap="viridis",
        s=10,
        alpha=0.75,
    )
    plt.colorbar(s, label=color_col)
    plt.xlabel("VAE latent z1")
    plt.ylabel("VAE latent z2")
    plt.title(title or f"Latent space colored by {color_col}")
    plt.show()


for col in INTERPRETATION_COLS:
    scatter_latent(col)

## 9. Latent regimes with simple clustering

To make the latent space more article-friendly, cluster the 2D points.

This lets us describe regimes like:

- high trend / low volatility
- high volatility stress
- mean-reversion pressure
- commodity-led macro regime

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=20)
analysis_df["latent_regime"] = kmeans.fit_predict(analysis_df[["z1", "z2"]])

plt.figure(figsize=(8, 6))
for regime, sub in analysis_df.groupby("latent_regime"):
    plt.scatter(sub["z1"], sub["z2"], s=10, alpha=0.65, label=f"Regime {regime}")
plt.xlabel("VAE latent z1")
plt.ylabel("VAE latent z2")
plt.title("VAE latent regimes")
plt.legend()
plt.show()

In [ ]:
regime_summary = (
    analysis_df
    .groupby("latent_regime")
    .agg(
        n_windows=("date", "count"),
        start_date=("date", "min"),
        end_date=("date", "max"),
        avg_trend_strength=("trend_strength", "mean"),
        avg_volatility_stress=("volatility_stress", "mean"),
        avg_mean_reversion_pressure=("mean_reversion_pressure", "mean"),
        avg_macro_commodity_regime=("macro_commodity_regime", "mean"),
        avg_forward_ret_5d=("forward_ret_5d", "mean"),
        avg_reconstruction_error=("reconstruction_error", "mean"),
    )
    .sort_values("n_windows", ascending=False)
)

regime_summary.round(4)

In [ ]:
def name_regime(row):
    labels = []
    if row["avg_volatility_stress"] == regime_summary["avg_volatility_stress"].max():
        labels.append("volatility-stress")
    if row["avg_trend_strength"] == regime_summary["avg_trend_strength"].max():
        labels.append("trend-strength")
    if abs(row["avg_mean_reversion_pressure"]) == regime_summary["avg_mean_reversion_pressure"].abs().max():
        labels.append("mean-reversion-pressure")
    if row["avg_macro_commodity_regime"] == regime_summary["avg_macro_commodity_regime"].max():
        labels.append("commodity-macro")
    return ", ".join(labels) if labels else "mixed / neutral"

article_regimes = regime_summary.copy()
article_regimes["plain_english_label"] = article_regimes.apply(name_regime, axis=1)
article_regimes[["plain_english_label", "n_windows", "avg_forward_ret_5d", "avg_reconstruction_error"]].round(4)

## 10. Reconstruction error as an anomaly signal

A high reconstruction error means:

> This market window was difficult for the VAE to reconstruct from its learned latent representation.

In finance, that can be interpreted cautiously as an unusual or less-familiar state.

For a trading or decision-support framework, high reconstruction error can support an `ABSTAIN` decision.

In [ ]:
top_anomalies = (
    analysis_df
    .sort_values("reconstruction_error", ascending=False)
    [["date", "reconstruction_error", "trend_strength", "volatility_stress", "mean_reversion_pressure", "macro_commodity_regime", "forward_ret_5d"]]
    .head(15)
)

top_anomalies.round(4)

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(analysis_df["date"], analysis_df["reconstruction_error"])
plt.title("VAE reconstruction error over time")
plt.xlabel("Date")
plt.ylabel("Reconstruction error")
plt.show()

## 11. Simple action interpretation

This is not a complete trading model. It is a simple bridge to your adaptive market-state framework.

We can assign a rough interpretation:

- High trend strength + low volatility stress → trend-following conditions
- High mean-reversion pressure + moderate volatility → possible mean-reversion conditions
- High volatility stress or high reconstruction error → uncertainty / abstention conditions
- High commodity regime score → commodity-led macro context

In [ ]:
def simple_state_label(row):
    vol_q = analysis_df["volatility_stress"].quantile(0.75)
    recon_q = analysis_df["reconstruction_error"].quantile(0.90)
    trend_q = analysis_df["trend_strength"].quantile(0.75)
    mr_q = analysis_df["mean_reversion_pressure"].abs().quantile(0.75)
    commodity_q = analysis_df["macro_commodity_regime"].quantile(0.75)

    if row["reconstruction_error"] >= recon_q or row["volatility_stress"] >= vol_q:
        return "ABSTAIN / STRESS"
    if abs(row["mean_reversion_pressure"]) >= mr_q:
        return "MEAN_REVERSION_PRESSURE"
    if row["trend_strength"] >= trend_q:
        return "TREND_STRENGTH"
    if row["macro_commodity_regime"] >= commodity_q:
        return "COMMODITY_MACRO"
    return "MIXED / NEUTRAL"

analysis_df["simple_state_label"] = analysis_df.apply(simple_state_label, axis=1)

state_summary = (
    analysis_df
    .groupby("simple_state_label")
    .agg(
        n_windows=("date", "count"),
        avg_forward_ret_5d=("forward_ret_5d", "mean"),
        avg_trend_strength=("trend_strength", "mean"),
        avg_volatility_stress=("volatility_stress", "mean"),
        avg_mean_reversion_pressure=("mean_reversion_pressure", "mean"),
        avg_macro_commodity_regime=("macro_commodity_regime", "mean"),
        avg_reconstruction_error=("reconstruction_error", "mean"),
    )
    .sort_values("n_windows", ascending=False)
)

state_summary.round(4)

In [ ]:
plt.figure(figsize=(8, 6))
for label, sub in analysis_df.groupby("simple_state_label"):
    plt.scatter(sub["z1"], sub["z2"], s=10, alpha=0.65, label=label)
plt.xlabel("VAE latent z1")
plt.ylabel("VAE latent z2")
plt.title("Latent space with simple market-state labels")
plt.legend(fontsize=8)
plt.show()

## 12. Article-friendly summary generator

This creates a first-pass written interpretation that can be edited into a Medium article.

In [ ]:
def make_article_summary(corr_table: pd.DataFrame, regime_summary: pd.DataFrame, state_summary: pd.DataFrame) -> str:
    strongest = []
    for z in ["z1", "z2"]:
        vals = corr_table.loc[z].dropna().abs().sort_values(ascending=False)
        if len(vals):
            strongest.append((z, vals.index[0], corr_table.loc[z, vals.index[0]]))

    lines = []
    lines.append("# Draft interpretation")
    lines.append("")
    lines.append("The VAE compresses rolling S&P 500 market windows into a two-dimensional latent space.")
    lines.append("Instead of treating the latent variables as direct forecasts, we interpret them as market-state coordinates.")
    lines.append("")

    for z, concept, corr in strongest:
        lines.append(f"- `{z}` is most associated with `{concept}` in this run, with correlation {corr:.3f}.")

    lines.append("")
    lines.append("The latent clusters show whether the model separates trend, stress, mean-reversion pressure, and commodity/macro context.")
    lines.append("High reconstruction-error windows can be treated as unusual regimes where an adaptive system may prefer to abstain rather than force a trend or mean-reversion decision.")
    lines.append("")
    lines.append("For a UCAIF-style paper, the key message is not that the VAE predicts returns by itself. The contribution is that a probabilistic latent representation can become an input to an adaptive decision framework that chooses between trend-following, momentum, mean-reversion, or abstention.")

    return "
".join(lines)


print(make_article_summary(correlation_table, regime_summary, state_summary))

## 13. How this connects to the agri ETF / UCAIF paper

This S&P 500 notebook is the intuition bridge.

For the agri ETF paper, replace the market universe with:

```python
TRADE_TICKERS = ("DBA", "WEAT", "CORN", "SOYB", "CANE")
CONTEXT_TICKERS = ("DBC", "UUP", "USO")
```

Then reuse the same conceptual pipeline:

```text
Rolling market window
    ↓
Trend / momentum / mean-reversion / volatility / macro features
    ↓
VAE latent representation
    ↓
Latent regime interpretation
    ↓
Adaptive action framework:
        FOLLOW_TREND
        FOLLOW_MOMENTUM
        MEAN_REVERT
        ABSTAIN
```

For publication, keep the claim modest:

> The VAE is not a standalone alpha model. It is a representation-learning tool for identifying market states and uncertainty.